PROJECT + Presentation: This is the case: you must develop using LLM and RAG. After submission you people must present to some of the panel members.
 Use case: “Policy & Claims Copilot” (Customer support + Claims pre-check)
Goal - Help customers, agents, and claims teams get instant, consistent answers about:

This solution demonstrates how to build a Policy & Claims Copilot using:

LLM (Large Language Model)
RAG (Retrieval Augmented Generation)
Policy PDF Documents
Vector Database (FAISS)
LangChain Framework
HuggingFace Embeddings
Gemini LLM.
                    User Question
                           │
                           ▼
                Policy & Claims Copilot
                           │
                           ▼
                  Query Processing
                           │
                           ▼
                 Vector Search (FAISS)
                           │
         Retrieves Relevant Policy Clauses
                           │
                           ▼
               Retrieved Policy Sections
                           │
                           ▼
                   LLM (Gemini)
                           │
                           ▼
      Grounded Response + Source Citation

In [ ]:
# Install requests,  to send web requests (to fetch data from the API)
!pip install requests==2.32.5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
# Install required Langchain and faiss ML libraries.
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-openai
!pip install -q pypdf
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q tiktoken
!pip install langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 50.0 MB/s eta 0:00:00


In [ ]:
# Upload the RAG data set containing Health insurance policy document
from google.colab import files

uploaded = files.upload()

Saving Health_Policy.pdf to Health_Policy (1).pdf


In [ ]:
# Read the PDF file "Health_Policy.pdf", split it by page, and count how many pages were successfully processed.
from langchain_community.document_loaders import TextLoader, PyPDFLoader

loader = PyPDFLoader("Health_Policy.pdf")

documents = loader.load()

print("Pages Loaded:", len(documents))

/tmp/ipykernel_494/184895192.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, PyPDFLoader


Pages Loaded: 11


In [ ]:
# Breaks the PDF file content into smaller, overlapping text blocks (chunks) to process the information efficiently.
# Chunk sixe  - 1000 characters and overlapping chunk of 150 chars to maintain the context between the chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

docs = splitter.split_documents(documents)

print("Chunks:", len(docs))

Chunks: 33


In [ ]:
!pip install langchain-huggingface

In [ ]:
# Use the below HF model to create a new model
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Convert the chunks into vectors (numbers), and save them into a searchable FAISS vector database on the local machine.
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores import Chroma

vectorstore = FAISS.from_documents(
    docs,
    embedding_model
)

vectorstore.save_local("policy_index")

In [ ]:
# Load  previously saved FAISS vector database back into memory from the computer's hard drive
db = FAISS.load_local(
    "policy_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

In [ ]:
!pip install -q langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 848.0 kB/s eta 0:00:00


In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = "--"

In [ ]:
# Connects this script to Google’s Gemini AI using the LangChain framework.
# Temparure of 0 makes the model completely deterministic, ensuring highly factual, consistent, and predictable answers.
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [ ]:
# Use FAISS database as a retriever engine, to fetch the top 3 most relevant text chunks for a question.
retriever = db.as_retriever(
    search_kwargs={"k":3}
)

In [ ]:
# Set the prompt template and context to answer a query using RAG
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

prompt = PromptTemplate(
    input_variables=["context","question"],
    template="""
You are an Insurance Policy Copilot.
Answer ONLY from the policy context.

If information is unavailable,
reply:
'Information not found in policy.'

Context:
{context}

Question:
{question}

Provide:
1. Answer
2. Source clause
3. Page number
"""
)

In [ ]:
# Use LangChain Expression Language (LCEL) to build a pipeline (chain) that connects the retriever, custom prompt, Gemini LLM,
# and an output cleaner into a single executable object

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Define your system prompt template
system_prompt = (
    "Use the given pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 2. Create the core LLM response chain
#question_answer_chain = create_stuff_documents_chain(llm, prompt)

# 3. Helper function to format list of documents into a single text block
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Build the RAG chain natively via LCEL
qa_chain = (
    {
        "context": retriever | format_docs,  # Retrieves docs and turns them to text
        "input": RunnablePassthrough()       # Passes the user question forward
    }
    | prompt                                 # Plugs context + question into the prompt
    | llm                                    # Sends the formatted prompt to your LLM
    | StrOutputParser()                      # Parses the output directly into a clean string
)


In [ ]:
# Test the model
#query = "Is maternity treatment covered?"
response = qa_chain.invoke("Is maternity treatment covered?")

# Access the generated text answer
print(response)

Yes, maternity treatment is covered. The policy covers medical expenses incurred for delivery, including pre-natal and post-natal complications.

However, there are some conditions and limitations:
*   **Waiting Period:** A continuous waiting period of twenty-four (24) months from the policy's inception date applies to all maternity claims, unless continuous coverage with an equivalent maternity rider was ported from a prior insurer without a gap.
*   **Maximum Deliveries:** Cover is limited strictly to the first two (2) living children born to the Insured Person during the lifetime of the policy.
*   **Financial Sub-Limits:** Normal delivery procedures are capped at $1,500 per event, and Caesarean Section (C-Section) operations are capped at $2,500 per event.
*   **Exclusions:** Regular outpatient consults, routine vitamin supplements, and non-indicated fetal ultrasounds are excluded from maternity coverage unless linked to a high-risk medical complication.


In [ ]:
# Test for hallucination
question = "What is LangChain?"

# 1. Fetch documents directly from the retriever component
# Note: If your retriever variable has a different name (e.g., base_retriever), use that here.
source_docs = retriever.invoke(question)

# 2. Safely loop through the retrieved documents
for doc in source_docs:
    metadata = getattr(doc, "metadata", {})
    page_num = metadata.get("page", "N/A")

    print("Page:", page_num)
    print(doc.page_content[:100])
    print("-" * 40)

    # Corrected comparison logic (handles strings like 'N/A' and checks valid page numbers)
    if isinstance(page_num, int) and page_num > 0:
        print(f"Page: {page_num}")
        print(doc.page_content[:50])
        print("-" * 40)
    else:
        print("No valid Page integer in the source_docs")

# 3. Print the actual text answer generated by the chain
result = qa_chain.invoke(question)
print("\nFinal Answer:", str(result))


Page: 0
13. SECTION 13: FRAUD DETECTION & POLICY VOIDANCE 
14. SECTION 14: PORTABILITY AND MIGRATION PROVISI
----------------------------------------
No valid Page integer in the source_docs
Page: 7
If the Insurer fails to resolve an un-deficient claim or delay disbursement beyond thirty (30) calen
----------------------------------------
Page: 7
If the Insurer fails to resolve an un-deficient cl
----------------------------------------
Page: 1
• Accident: A sudden, unforeseen, involuntary, and fortuitous event caused by external, 
visible, an
----------------------------------------
Page: 1
• Accident: A sudden, unforeseen, involuntary, and
----------------------------------------

Final Answer: I'm sorry, but the provided context does not contain any information about LangChain. Therefore, I cannot answer your question.


In [ ]:
# Check the page numbers in the RAG document where the context of the query is found

question = "Is maternity treatment covered?"

# 1. Get the underlying retriever from your chain setup
# (Replace `qa_chain.retriever` with your actual retriever variable if named differently)

# 2. Invoke the retriever directly to extract source documents
source_docs = retriever.invoke(question)

# 3. Print metadata and page content safely
for doc in source_docs:
    # Safely extract metadata dictionary
    metadata = getattr(doc, "metadata", {})
    page_num = metadata.get("page")

    print("Page:", page_num if page_num is not None else "N/A")
    print(doc.page_content[:100])
    print("-" * 40)

# 4. Generate the final answer using the full chain
result = qa_chain.invoke(question)
print("\nFinal Answer:", str(result))

Page: 4
• Included Interventions: Routine neonatal pediatric evaluations, mandatory immunization 
inoculatio
----------------------------------------
Page: 3
certified Rapid Antigen Test conducted in an authorized, government-approved laboratory. 
• Home Car
----------------------------------------
Page: 3
complications: 
• Maximum Admissible Deliveries: Cover is limited strictly to the first two (2) livi
----------------------------------------

Final Answer: Yes, maternity treatment is covered.

The policy covers medical expenses incurred for delivery, including pre-natal and post-natal complications. However, there are some conditions and limitations:

*   **Waiting Period:** A continuous waiting period of twenty-four (24) months from the inception date of the policy applies to all maternity claims.
*   **Maximum Deliveries:** Cover is limited strictly to the first two (2) living children born to the Insured Person during the lifetime of the policy.
*   **Financial Sub-Limits:** Nor

In [ ]:
# Test the model for a specific policy claim scenerio

claim_scenario = """
Hospitalized for dengue fever
for 4 days.
Policy taken 2 years ago.
Claim amount 95000.
"""

query = f"""
Check this claim before submission.

{claim_scenario}

Tell:
1. Likely covered?
2. Waiting period satisfied?
3. Required documents?
4. Missing information?
"""

# 1. Use modern .invoke() syntax
result = qa_chain.invoke(query)

# 2. Print the result directly (TextAccessor objects convert seamlessly to strings)
print(result)


Here's an assessment of the claim:

1.  **Likely covered?**
    No, it is **not likely covered** at this time. The policy was taken 2 years ago (24 months). According to the waiting period schedule, "Specific Diseases" (which includes Dengue Fever) are only covered from "25 Months to 48 Months." At 24 months, the policy is still in the period where "Specific Diseases & Maternity [are] Excluded."

2.  **Waiting period satisfied?**
    No, the waiting period for Specific Diseases has **not been satisfied**. The policy is 24 months old, and coverage for Specific Diseases begins at 25 months.

3.  **Required documents?**
    To initiate a valid evaluation, the following documents are mandatory:
    *   Duly Completed Claim Form (Form A and Form B), signed by both the Policyholder and the attending Medical Practitioner.
    *   Original Discharge Summary, explicitly detailing clinical history, presentation symptoms, physical examination findings, treatment course, and final prognosis.

4.  

In [ ]:
# Test search query against the FAISS database to find chunks about "cataract surgery," then loop through each matching text block
# to print out its page number and content.

docs = retriever.invoke("Waiting period for cataract surgery")

for d in docs:
    # 2. Defensively get the metadata dictionary
    metadata = getattr(d, "metadata", {})

    # 3. Use .get() to prevent KeyError if the "page" key is missing
    page_num = metadata.get("page", "N/A")

    print("Page:", page_num)
    print(d.page_content)
    print("-" * 40)  # Optional: separates documents visually


Page: 2
to the specific clinical condition for which the inpatient admission occurred. 
2.3 Post-Hospitalization Medical Expenses 
The Insurer will cover admissible medical expenses incurred during a period of exactly ninety (90) 
days immediately following the date of discharge from the hospital. These expenses must form a 
direct continuum of care for the illness or injury that prompted the initial hospitalization. 
2.4 Day Care Procedures 
The policy covers medical treatments and surgical procedures that require less than 24 hours of 
hospitalization due to technological advancements in modern medicine. This includes cataract 
surgeries, dialysis, chemotherapy, and minor corrective interventions, provided they are conducted 
under general or local anesthesia in a dedicated day care facility. 
2.5 Domiciliary Hospitalization 
Indemnity is extended for medical treatments undergone at home for a period exceeding three (3)
----------------------------------------
Page: 4
6. Experimental